# Step 5 — Strip Vision Encoder (Architecture + Weights)




In [ ]:
# ── Paths 

import os, json
from safetensors import safe_open
from safetensors.torch import save_file

OUT  = "/home/prashant.takale/icml/4_models/qwen3.5-4b-final-combo"
path = os.path.join(OUT, "model.safetensors")

assert os.path.exists(path), f"Step 4 output not found: {path}"
print("Paths OK.", flush=True)

In [ ]:
# ── Inspect: how many visual tensors exist before stripping? 

with safe_open(path, "pt") as f:
    all_keys_before = list(f.keys())

visual_before = [k for k in all_keys_before if "visual" in k or ".visual." in k]
print(f"Total tensors before   : {len(all_keys_before)}")
print(f"Visual tensors present : {len(visual_before)}")
print("Sample visual keys:", visual_before[:5])

In [ ]:
# ── Load and filter out visual tensors 

print("Loading combo model ...", flush=True)
kept, dropped = {}, 0
with safe_open(path, "pt") as f:
    meta = f.metadata()
    for k in f.keys():
        if "visual" in k or ".visual." in k:
            dropped += 1
            continue
        kept[k] = f.get_tensor(k)

print(f"  Kept {len(kept)} tensors, dropped {dropped} visual", flush=True)

In [ ]:
# ── Save stripped model 

kept = {k: v.clone().contiguous() for k, v in kept.items()}
print(f"Saving stripped model to {path} ...", flush=True)
save_file(kept, path, metadata=meta or {"format": "pt"})
print("Saved.", flush=True)

In [ ]:
# ── Clean config.json 

cfgp = os.path.join(OUT, "config.json")
cfg  = json.load(open(cfgp))

# Remove vision-specific top-level keys

for vk in ["vision_config", "image_token_id", "video_token_id",
           "vision_start_token_id", "vision_end_token_id"]:
    removed = cfg.pop(vk, None)
    if removed is not None:
        print(f"  Removed config key: {vk}")

# Remove visual entries from quantization ignore list

qc = cfg.get("quantization_config", {})
if "ignore" in qc:
    before = len(qc["ignore"])
    qc["ignore"] = [x for x in qc["ignore"] if "visual" not in x and "vision" not in x]
    print(f"  Quant ignore list: {before} → {len(qc['ignore'])} entries")

json.dump(cfg, open(cfgp, "w"), indent=2)
print("config.json cleaned.", flush=True)

In [ ]:
# ── Verify: no visual tensors remain 

with safe_open(path, "pt") as f:
    remaining_visual = sum(1 for k in f.keys() if "visual" in k)
    total = len(list(f.keys()))

print(f"Total tensors after strip : {total}")
print(f"Visual tensors remaining  : {remaining_visual}  (want 0)")

if remaining_visual == 0:
    print(f"\n Step 5 DONE — vision encoder fully removed → {OUT}")
else:
    print("\n❌ Visual tensors still present — check key filtering above!")

## Shim: How the runtime side works

The `shim/sitecustomize.py` file is placed on `PYTHONPATH` *before* vLLM starts (via `ENV PYTHONPATH=/opt/shim` in the Dockerfile). Python auto-imports `sitecustomize` at startup, which monkeypatches `Qwen3_5ForConditionalGeneration.__init__` to skip instantiating the vision tower and replace it with a zero-parameter `_NoVisual` stub.

```
# Container Dockerfile sets:
COPY shim/ /opt/shim/
ENV PYTHONPATH=/opt/shim
```

With both this notebook (weight strip) and the shim (architecture patch) applied:
- No visual **module** is built (shim)
- No visual **weights** exist (this notebook)
- The vision encoder is genuinely gone from the architecture